In [1]:
import pandas as pd
import requests
from io import BytesIO

# -------------------------------------------------------------------
# 1. URL oficial de descarga del Catálogo Extendido RECOGE (SIMBIO)
# -------------------------------------------------------------------
url = "https://simbio.mma.gob.cl/PlanesRecoge/GetExcelExtendido"

print("Descargando archivo Excel desde SIMBIO...")

# -------------------------------------------------------------------
# 2. Descargar el archivo directamente desde la web
# -------------------------------------------------------------------
response = requests.get(url)

# Verificar estado de descarga
if response.status_code != 200:
    raise Exception(
        f"Error al descargar el archivo. Código HTTP: {response.status_code}"
    )

print("Descarga completada correctamente ✅")

# -------------------------------------------------------------------
# 3. Leer Excel desde memoria (sin guardar en disco)
# -------------------------------------------------------------------
excel_data = BytesIO(response.content)

# -------------------------------------------------------------------
# 4. Definir hojas a cargar
# -------------------------------------------------------------------
sheets = {
    "decretos": "Decretos",
    "plan_simple": "Especies tipo plan simple",
    "plan_multiple": "Especiestipo plan múltiple",
}

# -------------------------------------------------------------------
# 5. Leer hojas en dataframes separados
# -------------------------------------------------------------------
df_decretos = pd.read_excel(excel_data, sheet_name=sheets["decretos"])
df_plan_simple = pd.read_excel(excel_data, sheet_name=sheets["plan_simple"])
df_plan_multiple = pd.read_excel(excel_data, sheet_name=sheets["plan_multiple"])

# -------------------------------------------------------------------
# 6. Inspección rápida
# -------------------------------------------------------------------
print("\n✅ Columnas - Decretos:")
print(df_decretos.columns)

print("\n✅ Columnas - Especies tipo plan simple:")
print(df_plan_simple.columns)

print("\n✅ Columnas - Especies tipo plan múltiple:")
print(df_plan_multiple.columns)

# -------------------------------------------------------------------
# 7. Mostrar primeras filas
# -------------------------------------------------------------------
print("\nPrimeras filas - Decretos:")
display(df_decretos.head())

print("\nPrimeras filas - Plan simple:")
display(df_plan_simple.head())

print("\nPrimeras filas - Plan múltiple:")
display(df_plan_multiple.head())


Descargando archivo Excel desde SIMBIO...
Descarga completada correctamente ✅

✅ Columnas - Decretos:
Index(['ID plan', 'N° Decreto', 'Nombre', 'Fecha promulgación',
       'Fecha publicación', 'URL'],
      dtype='str')

✅ Columnas - Especies tipo plan simple:
Index(['ID plan', 'Nombre común', 'Nombre científico',
       'Estado de conservación', 'Antecedentes de la especie'],
      dtype='str')

✅ Columnas - Especies tipo plan múltiple:
Index(['ID plan', 'Reino', 'Phyllum', 'Clase', 'Orden', 'Familia',
       'NombreComun', 'NombreCientifico', 'EstadoConservacion',
       'UrlFichaInventario'],
      dtype='str')

Primeras filas - Decretos:


,ID plan,N° Decreto,Nombre,Fecha promulgación,Fecha publicación,URL
0,RECOGE0001,12345,Decreto Test,01/09/2020,11/09/2020,https://www.soporta.cl
1,RECOGE0001,NS34,Otro Decreto,07/05/2020,01/06/2020,https://www.google.com
2,RECOGE0001,Por definir,Por definir,02/01/2020,01/01/2020,https://www.google.cl
3,RECOGE0001,Por definir,Por definir,01/01/2019,01/01/2019,por definir
4,RECOGE0001,Por definir,Por definir,01/01/2021,01/01/2021,Por definir



Primeras filas - Plan simple:


,ID plan,Nombre común,Nombre científico,Estado de conservación,Antecedentes de la especie
0,RECOGE0002,Ruil,Nothofagus alessandrii,En Peligro (EN),El ruil (Nothofagus alessandrii) es un árbol q...
1,RECOGE0003,Lucumillo,Myrcianthes coquimbensis,En Peligro (EN),"El lucumillo es un arbusto de copa globosa, re..."
2,RECOGE0004,Garra de león,Leontochir ovallei,En Peligro (EN),La garra de león o Leontochir ovallei (...
3,RECOGE0005,Chinchilla de cola corta,"Chinchilla chinchilla (Lichtenstein, 1829)",En peligro crítico (CR),La taxonomía del género Chinchilla ha dado pie...
4,RECOGE0006,Huemul,Hippocamelus bisulcus (Molina 1782),En Peligro (EN),El huemul fue descrito en 1782 por Juan Ignaci...



Primeras filas - Plan múltiple:


,ID plan,Reino,Phyllum,Clase,Orden,Familia,NombreComun,NombreCientifico,EstadoConservacion,UrlFichaInventario
0,RECOGE0001,Plantae,Magnoliophyta,Magnoliopsida,Fabales,Fabaceae,.,Adesmia micrantha,VU,https://simbio.mma.gob.cl/Especies/Details/13237
1,RECOGE0001,Plantae,Pteridophyta,Polypodiopsida,Polypodiales,Pteridaceae,Palito negro,Adiantum chilense,LC,https://simbio.mma.gob.cl/Especies/Details/15749
2,RECOGE0001,Plantae,Magnoliophyta,Liliopsida,Liliales,Alstroemeriaceae,Lirio,Alstroemeria graminea,VU,https://simbio.mma.gob.cl/Especies/Details/6925
3,RECOGE0001,Plantae,Magnoliophyta,Liliopsida,Liliales,Alstroemeriaceae,Lirio,Alstroemeria lutea,EN,https://simbio.mma.gob.cl/Especies/Details/6934
4,RECOGE0001,Plantae,Pteridophyta,Polypodiopsida,Polypodiales,Aspleniaceae,Helecho,Asplenium peruvianum,CR,https://simbio.mma.gob.cl/Especies/Details/17728


In [3]:
import pandas as pd
import numpy as np

# --- Asumo que ya tienes cargados estos dataframes ---
# df_decretos
# df_plan_simple
# df_plan_multiple

# =========================
# 1) Identificar columnas clave (robusto a variaciones de nombres)
# =========================
def pick_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        for k, v in cols.items():
            if cand in k:
                return v
    return None

# En "Decretos"
col_fecha = pick_col(df_decretos, ["fecha public", "fecha_pub", "public"])
col_idplan = pick_col(df_decretos, ["id plan", "id_plan", "idplan", "plan id", "id del plan", "id"])
# En "Decretos" (agregar detección de N° Decreto)
col_decreto = pick_col(df_decretos, [
    "n° decreto", "nº decreto", "numero decreto", "nro decreto", "no decreto",
    "decreto n", "decreto", "num decreto", "n decreto"
])

if col_decreto is None:
    raise ValueError(
        "No encontré una columna de 'N° Decreto' (o similar) en la hoja 'Decretos'. "
        f"Columnas disponibles: {list(df_decretos.columns)}"
    )

if col_fecha is None:
    raise ValueError("No encontré una columna de 'Fecha promulgación' (o similar) en la hoja 'Decretos'.")
if col_idplan is None:
    raise ValueError("No encontré una columna de ID de plan (ID Plan) en la hoja 'Decretos'.")

# En "Especies tipo plan simple"
col_idplan_simple = pick_col(df_plan_simple, ["id plan", "id_plan", "idplan", "plan id", "id del plan", "id"])
col_nombre_simple = pick_col(df_plan_simple, ["nombre científico", "nombre_cient", "nombrecient", "scientific"])

if col_idplan_simple is None or col_nombre_simple is None:
    raise ValueError("No encontré columnas (ID Plan / Nombre científico) en 'Especies tipo plan simple'.")

# En "Especiestipo plan múltiple"
col_idplan_mult = pick_col(df_plan_multiple, ["id plan", "id_plan", "idplan", "plan id", "id del plan", "id"])
col_nombre_mult = pick_col(df_plan_multiple, ["nombre científico", "nombre_cient", "nombrecient", "scientific", "nombrecientifico", "nombre cientifico"])

if col_idplan_mult is None or col_nombre_mult is None:
    raise ValueError("No encontré columnas (ID Plan / NombreCientifico) en 'Especiestipo plan múltiple'.")


# =========================
# 2) Limpiar Decretos: excluir SOLO filas inválidas de N° Decreto
# =========================
excluir_decretos = {"12345", "NS34", "Por definir"}

df_decretos = df_decretos.copy()

df_decretos[col_decreto] = (
    df_decretos[col_decreto]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# 👉 EXCLUYE SOLO LAS FILAS inválidas
df_decretos = df_decretos.loc[
    ~df_decretos[col_decreto].isin(excluir_decretos)
]

# =========================
# 2) Normalizar / limpiar fechas e IDs
# =========================
df_decretos = df_decretos.copy()
df_decretos[col_fecha] = pd.to_datetime(df_decretos[col_fecha], errors="coerce")
df_decretos["Año"] = df_decretos[col_fecha].dt.year

# Normalizar ID plan como string (para cruces consistentes)
def norm_id(s):
    return s.astype(str).str.strip()

df_decretos["_ID_PLAN"] = norm_id(df_decretos[col_idplan])
df_plan_simple = df_plan_simple.copy()
df_plan_multiple = df_plan_multiple.copy()

df_plan_simple["_ID_PLAN"] = norm_id(df_plan_simple[col_idplan_simple])
df_plan_multiple["_ID_PLAN"] = norm_id(df_plan_multiple[col_idplan_mult])

# =========================
# 3) Para cada ID plan: especie única (simple) y número de especies (múltiple)
# =========================
# --- Plan simple: 1 especie por plan (nos quedamos con el nombre, y conteo=1) ---
simple_por_plan = (
    df_plan_simple
    .dropna(subset=[col_nombre_simple])
    .groupby("_ID_PLAN")[col_nombre_simple]
    .agg(
        Especie_simple=lambda x: x.dropna().astype(str).iloc[0] if len(x.dropna()) else np.nan,
        N_especies_simple=lambda x: x.dropna().astype(str).nunique()
    )
    .reset_index()
)

# --- Plan múltiple: contar especies únicas por plan ---
mult_por_plan = (
    df_plan_multiple
    .dropna(subset=[col_nombre_mult])
    .groupby("_ID_PLAN")[col_nombre_mult]
    .agg(N_especies_multiple=lambda x: x.dropna().astype(str).nunique())
    .reset_index()
)

# =========================
# 4) Unir al maestro de decretos (por ID plan) y construir tabla por año
# =========================
planes = (
    df_decretos[["_ID_PLAN", "Año"]]
    .dropna(subset=["Año"])
    .drop_duplicates()
    .merge(simple_por_plan, on="_ID_PLAN", how="left")
    .merge(mult_por_plan, on="_ID_PLAN", how="left")
)

# Si un plan no aparece en simple/múltiple, sus conteos quedan NaN -> 0
planes["N_especies_simple"] = planes["N_especies_simple"].fillna(0).astype(int)
planes["N_especies_multiple"] = planes["N_especies_multiple"].fillna(0).astype(int)

# Tabla por año
planes_por_año = (
    planes.groupby("Año", as_index=False)
    .agg(
        **{
            "Nuevos Planes RECOGE": ("_ID_PLAN", "nunique"),
            "Especies en planes simples": ("N_especies_simple", "sum"),
            "Especies en planes múltiples": ("N_especies_multiple", "sum"),
        }
    )
    .sort_values("Año")
)

# Total especies consideradas por año (simple + múltiple)
planes_por_año["Especies totales (simple+múltiple)"] = (
    planes_por_año["Especies en planes simples"] + planes_por_año["Especies en planes múltiples"]
)

# Acumulados
planes_por_año["Acumulado planes"] = planes_por_año["Nuevos Planes RECOGE"].cumsum()
planes_por_año["Acumulado especies (simple+múltiple)"] = planes_por_año["Especies totales (simple+múltiple)"].cumsum()

# Mostrar tabla final
print(planes_por_año)


    Año  Nuevos Planes RECOGE  Especies en planes simples  \
0  2018                     3                           2   
1  2021                     3                           3   
2  2022                     6                           3   
3  2023                     1                           1   
4  2024                     1                           1   
5  2025                     2                           2   

   Especies en planes múltiples  Especies totales (simple+múltiple)  \
0                            92                                  94   
1                             0                                   3   
2                            12                                  15   
3                             0                                   1   
4                             0                                   1   
5                             0                                   2   

   Acumulado planes  Acumulado especies (simple+múltiple)  
0             

/tmp/ipykernel_8558/3559436171.py:78: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_decretos[col_fecha] = pd.to_datetime(df_decretos[col_fecha], errors="coerce")


In [5]:
# Ruta de salida
# Ruta de salida
output_path = (
    "/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/"
    "MN6_RECOGE/planes_RECOGE_por_anio.xlsx"
)

# Exportar tabla a Excel
planes_por_año.to_excel(output_path, index=False)

# Confirmación en pantalla
print("✅ Exportación completada correctamente")
print(f"📄 Archivo guardado en:\n{output_path}")
print(f"📊 Filas exportadas: {planes_por_año.shape[0]}")
print(f"📌 Columnas exportadas: {planes_por_año.shape[1]}")

# Mostrar preview en notebook
display(planes_por_año.head())


✅ Exportación completada correctamente
📄 Archivo guardado en:
/home/ricardo/Documents/IEB/BIODATA/vinculaciones/PNUD/7NR_ejecucion/MN6_RECOGE/planes_RECOGE_por_anio.xlsx
📊 Filas exportadas: 6
📌 Columnas exportadas: 7


,Año,Nuevos Planes RECOGE,Especies en planes simples,Especies en planes múltiples,Especies totales (simple+múltiple),Acumulado planes,Acumulado especies (simple+múltiple)
0,2018,3,2,92,94,3,94
1,2021,3,3,0,3,6,97
2,2022,6,3,12,15,12,112
3,2023,1,1,0,1,13,113
4,2024,1,1,0,1,14,114
